In [7]:
import torch
assert torch.cuda.is_available(), "GPU bağlı değil. egitime başlama"

device = torch.device("cuda")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    "GB",
)

PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU: NVIDIA A100-SXM4-40GB
GPU VRAM: 39.5 GB


In [8]:
!nvidia-smi

Mon Aug 17 11:21:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [9]:
import os 
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path("/content/fire_guncel_repo")
CODE_ARCHIVE = Path("/content/eytnet_code.tar")

assert CODE_ARCHIVE.is_file(), (
    f"Kod arşivi bulunamadı: {CODE_ARCHIVE}\n"
    "VS Code'da eytnet_code.tar dosyasına sağ tıklayıp "
    "'Upload to Colab' seç."
)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        "tar",
        "-xf",
        str(CODE_ARCHIVE),
        "-C",
        str(PROJECT_ROOT)
    ],
    check=True
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Proje kökü:", PROJECT_ROOT)
print("Klasörler:", sorted(p.name for p in PROJECT_ROOT.iterdir()))

Proje kökü: /content/fire_guncel_repo
Klasörler: ['configs', 'eytnet', 'helpers']


In [10]:
%pip install -q -U kagglehub

In [11]:
import time
import kagglehub

DATA_ROOT = PROJECT_ROOT / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

expected_counts = {
    "train": 3900,
    "val": 1300,
    "test": 1300,
}

def dataset_is_ready():
    for split, expected in expected_counts.items():
        image_dir = DATA_ROOT / split / "images"
        label_dir = DATA_ROOT / split / "labels"

        if not image_dir.is_dir() or not label_dir.is_dir():
            return False

        image_count = len(list(image_dir.glob("*.jpg")))
        label_count = len(list(label_dir.glob("*.txt")))

        if image_count != expected or label_count != expected:
            return False

    return True

if dataset_is_ready():
    print("Dataset zaten hazır; yeniden indirilmiyor.")
else:
    started = time.time()

    downloaded_path = kagglehub.dataset_download(
        "pengbo00/home-fire-dataset",
        output_dir=str(DATA_ROOT),
        force_download=True,
    )

    print("Dataset yolu:", downloaded_path)
    print(
        "İndirme ve çıkarma süresi:",
        round(time.time() - started, 1),
        "saniye",
    )

100%|██████████| 1.70G/1.70G [00:09<00:00, 197MB/s]

Extracting files...


Dataset yolu: /content/fire_guncel_repo/data
İndirme ve çıkarma süresi: 41.2 saniye


In [12]:
for split, expected in expected_counts.items():
    image_dir = DATA_ROOT / split / "images"
    label_dir = DATA_ROOT / split / "labels"

    image_count = len(list(image_dir.glob("*.jpg")))
    label_count = len(list(label_dir.glob("*.txt")))

    print(
        f"{split:5s} | images={image_count} | labels={label_count}"
    )

    assert image_count == expected, (
        f"{split} görüntü sayısı yanlış: "
        f"{image_count}, beklenen: {expected}"
    )

    assert label_count == expected, (
        f"{split} etiket sayısı yanlış: "
        f"{label_count}, beklenen: {expected}"
    )

print("Dataset eksiksiz.")

train | images=3900 | labels=3900
val   | images=1300 | labels=1300
test  | images=1300 | labels=1300
Dataset eksiksiz.


In [13]:
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from eytnet.config import Config
from eytnet.dataset import build_dataloader
from eytnet.model import build_model
from eytnet.loss import EYTNetLoss
from eytnet.train import train

print("Tüm importlar başarılı.")

Tüm importlar başarılı.


In [14]:
cfg_test = Config.load("configs/experiment.json")
train_loader_test = build_dataloader(cfg_test, "train")
images, targets, metas = next(iter(train_loader_test))

print("Images:", images.shape, images.dtype)
print("Targets:", targets.shape)
print("Metas:", len(metas))
print("Min/max:", images.min().item(), images.max().item())

assert images.shape == (cfg_test.batch_size,3,640,640)
assert len(metas) == cfg_test.batch_size
assert images.min().item() >= 0.0
assert images.max().item() <= 1.0

del train_loader_test, images, targets, metas
print("dataloader test başarılı.")



Images: torch.Size([16, 3, 640, 640]) torch.float32
Targets: torch.Size([20, 6])
Metas: 16
Min/max: 0.0 1.0
dataloader test başarılı.


In [15]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
DRIVE_RUNS = Path("/content/drive/MyDrive/EYTNet/runs")
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

def load_colab_config(config_stem):
    cfg = Config.load(f"configs/{config_stem}.json")

    # sadece calısan config nesnesinin çıktı yolu değişioyr repo içinde json değişmiyor
    cfg.run_dir = DRIVE_RUNS / cfg.experiment_name

    return cfg

print("kalıcı sonuç klasörü", DRIVE_RUNS)



kalıcı sonuç klasörü /content/drive/MyDrive/EYTNet/runs


In [ ]:
cfg = load_colab_config("experiment")

last_checkpoint = cfg.run_dir / "weights" / "last.pt"
resume = last_checkpoint.exists()

print("Deney:", cfg.experiment_name)
print("epoch sayısı:", cfg.epochs)
print("Batch size:", cfg.batch_size)
print("GPU:", torch.cuda.get_device_name(0))
print("Çıktı klasörü:", cfg.run_dir)
print("Checkpoint'ten devam:", resume)

model, run_dir, history = train(
    cfg,
    device=device,
    resume=resume,
)


Deney: eytnet_baseline
epoch sayısı: 30
Batch size: 16
GPU: NVIDIA A100-SXM4-40GB
Çıktı klasörü: /content/drive/MyDrive/EYTNet/runs/eytnet_baseline
Checkpoint'ten devam: False
eytnet_baseline | 7,208,138 parametre | cuda
  [50/243] loss=6.5550 lr=0.00007
  [100/243] loss=6.3023 lr=0.00014
  [150/243] loss=6.1263 lr=0.00020
  [200/243] loss=5.9874 lr=0.00027
epoch 1/30 | loss 5.8351 | mAP@0.5 0.0045 | F2 0.0000 | 92.5s
  [50/243] loss=5.0610 lr=0.00040
  [100/243] loss=5.0582 lr=0.00047
  [150/243] loss=4.9995 lr=0.00054
  [200/243] loss=4.9921 lr=0.00061
epoch 2/30 | loss 4.9566 | mAP@0.5 0.0068 | F2 0.0000 | 93.0s
  [50/243] loss=4.7440 lr=0.00073
  [100/243] loss=4.7027 lr=0.00080
  [150/243] loss=4.6887 lr=0.00087
  [200/243] loss=4.6778 lr=0.00094
epoch 3/30 | loss 4.6638 | mAP@0.5 0.0344 | F2 0.0000 | 157.1s
  [50/243] loss=4.5968 lr=0.00100
  [100/243] loss=4.5737 lr=0.00100
  [150/243] loss=4.5296 lr=0.00100
  [200/243] loss=4.5147 lr=0.00100
epoch 4/30 | loss 4.4845 | mAP@0.5 0

In [ ]:
df = pd.read_csv(run_dir / "metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df["epoch"], df["train_loss"]); axes[0].set_title("train loss")
axes[1].plot(df["epoch"], df["map50"], marker="o"); axes[1].set_title("val mAP@0.5")
for ax in axes: ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
for k in ["box", "obj", "cls"]:
    plt.plot(df["epoch"], df[k], label=k)
plt.xlabel("epoch"); plt.ylabel("kayip"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
cfg_noaug = load_colab_config("experiment_no_augmentation")
last_checkpoint_noaug = (
    cfg_noaug.run_dir / "weights" / "last.pt"
)
resume_noaug = last_checkpoint_noaug.exists()

print("Deney:", cfg_noaug.experiment_name)
print("Checkpoint'ten devam:", resume_noaug)

model, run_dir_noaug, history_noaug = train(
    cfg_noaug,
    device=device,
    resume=resume_noaug,
)

In [ ]:
cfg_sgd = load_colab_config("experiment_sgd")

last_checkpoint_sgd = (
    cfg_sgd.run_dir / "weights" / "last.pt"
)
resume_sgd = last_checkpoint_sgd.exists()

print("Deney:", cfg_sgd.experiment_name)
print("Checkpoint'ten devam:", resume_sgd)

model, run_dir_sgd, history_sgd = train(
    cfg_sgd,
    device=device,
    resume=resume_sgd,
)

In [ ]:
cfg_lr5e4 = load_colab_config("experiment_adamw_lr5e4")

last_checkpoint_lr5e4 = (
    cfg_lr5e4.run_dir / "weights" / "last.pt"
)
resume_lr5e4 = last_checkpoint_lr5e4.exists()

print("Deney:", cfg_lr5e4.experiment_name)
print("Checkpoint'ten devam:", resume_lr5e4)

model, run_dir_lr5e4, history_lr5e4 = train(
    cfg_lr5e4,
    device=device,
    resume=resume_lr5e4,
)

In [ ]:
satirlar = []

deneyler = [
    "eytnet_baseline",
    "eytnet_no_augmentation",
    "eytnet_sgd",
    "eytnet_adamw_lr5e4",
]

for isim in deneyler:
    yol = DRIVE_RUNS / isim / "metrics.csv"

    if not yol.exists():
        print(f"Uyarı: {isim} için metrics.csv bulunamadı.")
        continue

    d = pd.read_csv(yol)

    if d.empty:
        print("atlandı, metrics bos", yol)
        continue

    best = d.loc[d["map50"].idxmax()]

    satirlar.append(
        {
            "deney": isim.replace("eytnet_", ""),
            "epoch": int(best["epoch"]),
            "map50": round(float(best["map50"]), 4),
            "precision": round(float(best["precision"]), 4),
            "recall": round(float(best["recall"]), 4),
            "f2": round(float(best["f2"]), 4),
        }
    )

karsilastirma = pd.DataFrame(satirlar)

if karsilastirma.empty:
    print("Henüz karşılaştırılacak tamamlanmış deney yok.")
else:
    karsilastirma = (
        karsilastirma
        .sort_values("map50", ascending=False)
        .reset_index(drop=True)
    )

    display(karsilastirma)

In [ ]:
cfg_final = load_colab_config("experiment_final")

last_checkpoint_final = (
    cfg_final.run_dir / "weights" / "last.pt"
)
resume_final = last_checkpoint_final.exists()

print("Final deney:", cfg_final.experiment_name)
print("Epoch:", cfg_final.epochs)
print("Çıktı klasörü:", cfg_final.run_dir)
print("Checkpoint'ten devam:", resume_final)

model, run_dir_final, history_final = train(
    cfg_final,
    device=device,
    resume=resume_final,
)

In [ ]:
df_final = pd.read_csv(run_dir_final / "metrics.csv")

fig, axes = plt.subplots(1,2,figsize=(12, 4))

axes[0].plot(
    df_final["epoch"],
    df_final["train_loss"],
)
axes[0].set_title("final train loss")
axes[1].plot(
    df_final["epoch"],
    df_final["map50"],
    marker="o",
)

axes[1].set_title("final val mAP@0.5")

for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
best_final = df_final.loc[df_final["map50"].idxmax()]

print("En iyi epoch:", int(best_final["epoch"]))
print("En iyi mAP@0.5:", round(float(best_final["map50"]), 4))
print("Precision:", round(float(best_final["precision"]), 4))
print("Recall:", round(float(best_final["recall"]), 4))
print("F2:", round(float(best_final["f2"]), 4))
print("Best checkpoint:", run_dir_final / "weights" / "best.pt")